<div align="center">

# Universidad de San Martín

## Infraestructura para Ciencia de Datos

### Licenciatura en Ciencia de Datos

<img src="../../logo.jpg" alt="Logo UNSAM" width="300"/>

---

</div>

# Ejercicio 06: El veredicto

---

## Objetivo
Practicar lo que hace alguien de MLOps cuando le toca decidir: **consultar el tracking y elegir**. Un equipo dejó **siete candidatos** registrados en MLflow, con sus métricas y nada más — ninguno viene con recomendación. Tu trabajo es mirarlos y responder **cinco preguntas** sobre ellos, una de ellas **cuál iría a producción**.

> **Acá no se entrena nada.** Los candidatos ya están registrados: lo que practicás es **consultar MLflow** y leer lo que devuelve.

> **Un solo archivo, autocontenido**: **Parte 1** deja los candidatos en el tracking y te muestra cómo consultarlos; **Parte 2** son cinco preguntas. Al final, **📦 Entrega** genera tu `.txt`.

> **Nota:** usa el **MLflow del stack** (`localhost:5000`) y, si no responde, cae a una carpeta local. Si MLflow no está instalado, la tabla sale igual, directo de los candidatos: las preguntas se contestan igual. El `.txt` registra cuál se usó.

> Las respuestas **no se autocorrigen**: las lee el docente en tu PR.

---

> **Sobre la entrega**: leé [`README.md`](README.md) en esta carpeta — el deliverable es **un `.txt` por estudiante** en `estudiantes/`, **no el notebook**.

## ⚠️ Antes de empezar: tu rama, con el material de esta clase

Para subir tu entrega tenés que estar parado en **tu rama personal** (la que creaste en la Clase 01: `estudiante/apellido-nombre`), con el material nuevo de `main` adentro. Si estás en `main` o `dev`, vas a romper el patrón de PR del curso.

**Recordá cómo se arma**: minúsculas, sin tildes y **un solo guión** separando apellido de nombre; los compuestos van pegados (`estudiante/garcialopez-mariajose`). Es la misma rama de la Clase 01: no crees una nueva.

```bash
# Ver en qué rama estás
git branch --show-current

# Al empezar CADA clase: traer lo nuevo de main a tu rama
git checkout main
git pull origin main
git checkout estudiante/apellido-nombre   # reemplazá por tu rama
git merge main --no-edit
```

> Si abriste esta notebook estando en `main`, no te cambies a tu rama sin el `merge`: la notebook todavía no está en tu rama. Cerrala, corré los cuatro comandos y volvé a abrirla.

> La sección **📦 Entrega** del final verifica esto programáticamente y te avisa si estás en una rama no esperada.

---
# Parte 1 — Los candidatos (una sola vez)

**¿Qué hacen estas celdas?**
1. Dejan en MLflow un experimento `clase06_veredicto` con **siete candidatos**, cada uno con sus params y sus métricas.
2. Los **consultan** de tres formas: todos juntos, filtrados y ordenados del lado del server, y en la UI.

**No tenés que tocar nada.** Solo correrlas en orden.

> Los candidatos son **casos de estudio**: un run de MLflow es una fila con params y métricas, y el modelo serializado es opcional — acá no hay `fit`, ni dataset, ni artefacto. Es el tablero que vas a mirar, no un entrenamiento.

In [ ]:
# ============================================================
# Los siete candidatos, cargados al tracking.
# ============================================================
# OJO: son CASOS DE ESTUDIO, no entrenamientos. Un run de MLflow es una fila
# con params y metricas -- el modelo serializado es opcional y aca no existe:
# no hay fit, no hay dataset, no hay artefacto.
#
# La celda es idempotente: si los candidatos ya estan, no los vuelve a crear.
EXPERIMENTO = "clase06_veredicto"

CANDIDATOS = {
    #                         algoritmo             vent  n_tr n_te  train test
    "random_forest_7d":      ("RandomForest",        7,   240,  60, 0.71, 0.68),
    "random_forest_3d_deep": ("RandomForest",        3,   240,  60, 0.98, 0.54),
    "decision_tree_1d":      ("DecisionTree",        1,    12,   3, 0.88, 0.83),
    "logistic_3d":           ("LogisticRegression",  3,   240,  60, 0.67, 0.66),
    "gradient_boosting_7d":  ("GradientBoosting",    7,   240,  60, 0.64, 0.61),
    "svm_3d":                ("SVM",                 3,   240,  60, 0.52, 0.51),
    "random_forest_7d_v2":   ("RandomForest",        7,   240,  60, 0.74, 0.69),
}

# Donde quedan, de mejor a peor (el .txt de la entrega registra cual se uso):
#   1) el MLflow del stack, si responde en localhost:5000;
#   2) si no responde, una carpeta local;
#   3) si MLflow no esta instalado (o falla por lo que sea), en ningun lado:
#      la tabla sale directo de CANDIDATOS y las preguntas se contestan igual.
try:
    import mlflow
    import requests
    try:
        requests.get("http://localhost:5000/health", timeout=5).raise_for_status()
        TRACKING, MODO = "http://localhost:5000", "server"
    except Exception as e:
        TRACKING, MODO = "file:./mlruns_veredicto", "local"
        print(f"El server de MLflow no respondio ({type(e).__name__}): uso una carpeta local.")
        print("Si esperabas usar el del stack, revisa que este levantado: docker compose ps")
    mlflow.set_tracking_uri(TRACKING)
    mlflow.set_experiment(EXPERIMENTO)

    cliente = mlflow.MlflowClient()
    exp_id = cliente.get_experiment_by_name(EXPERIMENTO).experiment_id
    ya_estan = {r.info.run_name for r in cliente.search_runs([exp_id])}

    for nombre_run, (algo, vent, n_tr, n_te, tr, te) in CANDIDATOS.items():
        if nombre_run in ya_estan:
            continue
        with mlflow.start_run(run_name=nombre_run):
            mlflow.set_tag("caso_de_estudio", "clase06_veredicto")
            mlflow.log_param("algoritmo", algo)
            mlflow.log_param("ventana_dias", vent)
            mlflow.log_param("n_fechas_train", n_tr)
            mlflow.log_param("n_fechas_test", n_te)
            mlflow.log_metric("accuracy_train", tr)
            mlflow.log_metric("accuracy_test", te)

    nuevos = len(CANDIDATOS) - len(ya_estan & set(CANDIDATOS))
    print(f"Candidatos en el experimento: {len(CANDIDATOS)}  (creados ahora: {nuevos})")
except Exception as e:
    mlflow = None
    TRACKING, MODO = "-", "sin_mlflow"
    if isinstance(e, ImportError):
        print("MLflow no esta instalado en este entorno: la tabla sale directo de los candidatos.")
        print("Para usarlo: pip install -r requirements.txt desde la raiz del repo, y reinicia el kernel.")
    else:
        print(f"MLflow fallo ({type(e).__name__}: {e}).")
        print("La tabla sale directo de los candidatos.")
    print("Las preguntas se contestan igual.")
print(f"Tracking: {MODO} ({TRACKING})")

---
## Consultar el tracking

Tres formas de mirar lo mismo, y conviene tenerlas a mano:

- **`search_runs`** devuelve un DataFrame con params y métricas, una fila por run.
- El **mismo pedido filtrando y ordenando del lado del server** (`filter_string`, `order_by`): con cientos de runs no se bajan todos para mirarlos a mano.
- La **UI** (`localhost:5000`): seleccionar varios runs y compararlos, que es como se hace en el día a día.

> Si MLflow no está instalado, la celda arma la misma tabla con pandas: los números son los mismos.

In [ ]:
# ============================================================
# Consultar el tracking: esto es lo que se practica.
# ============================================================
import pandas as pd

COLUMNAS = ["tags.mlflow.runName", "params.n_fechas_train", "params.n_fechas_test",
            "metrics.accuracy_train", "metrics.accuracy_test"]
CORTO = ["candidato", "n_train", "n_test", "acc_train", "acc_test"]


def leer_candidatos():
    """Los siete candidatos como tabla, ordenados por nombre.

    Con MLflow, lo que devuelve el tracking. Sin MLflow, directo de CANDIDATOS:
    los mismos numeros (n_train y n_test como texto, igual que los params de MLflow).
    """
    if mlflow is not None:
        t = mlflow.search_runs(experiment_names=[EXPERIMENTO])[COLUMNAS].set_axis(CORTO, axis=1)
    else:
        t = pd.DataFrame([(n, str(v[2]), str(v[3]), v[4], v[5]) for n, v in CANDIDATOS.items()],
                         columns=CORTO)
    return t.sort_values("candidato").reset_index(drop=True)


# 1) Todos los runs del experimento, como DataFrame.
tabla = leer_candidatos()
print("Los siete candidatos:")
display(tabla)

# 2) El mismo pedido, pero filtrando y ordenando DEL LADO DEL SERVER: con cientos
#    de runs no se bajan todos para mirarlos a mano.
if mlflow is not None:
    mejores = mlflow.search_runs(
        experiment_names=[EXPERIMENTO],
        filter_string="metrics.accuracy_test > 0.6",
        order_by=["metrics.accuracy_test DESC"],
    )[COLUMNAS].set_axis(CORTO, axis=1)
else:
    # Sin MLflow: el mismo filtro y el mismo orden, con pandas.
    mejores = tabla[tabla["acc_test"] > 0.6].sort_values("acc_test", ascending=False)
print("\nLos que pasan accuracy_test > 0.6, ordenados por accuracy_test:")
display(mejores.reset_index(drop=True))

# 3) Y lo mismo se mira en la UI: http://localhost:5000 -> experimento
#    "clase06_veredicto" -> seleccionar varios runs -> "Compare".

---
# Parte 2 — Cinco preguntas

Respondé estas cinco **con tus palabras** en la celda de abajo. Corto está bien. Van incluidas en tu `.txt` y **las lee el docente en tu PR**: no se autocorrigen.

Qué es cada columna de la tabla de candidatos:

| Columna | Qué es |
| :--- | :--- |
| `n_train` / `n_test` | cuántas fechas se usaron para entrenar y para medir |
| `acc_train` | qué parte acierta sobre los datos con los que se entrenó |
| `acc_test` | qué parte acierta sobre datos que no vio al entrenar |

## Las preguntas

| # | Pregunta |
| :---: | :--- |
| a | ¿Qué podés decir del modelo con el `acc_test` más alto? |
| b | ¿Qué podés decir del modelo con el `acc_train` más alto? |
| c | ¿Qué podés decir del modelo con el `acc_test` más bajo? |
| d | ¿Qué candidato pondrías en producción? ¿Por qué? |
| e | Con el modelo que elegiste ya en producción, ¿cómo te darías cuenta de que empezó a andar peor? |

In [ ]:
# Respondé DENTRO de las comillas, con tus palabras. Corto está bien.

# (a) ¿Qué podés decir del modelo con el acc_test más alto?
respuesta_a = ''

# (b) ¿Qué podés decir del modelo con el acc_train más alto?
respuesta_b = ''

# (c) ¿Qué podés decir del modelo con el acc_test más bajo?
respuesta_c = ''

# (d) ¿Qué candidato pondrías en producción? ¿Por qué?
candidato = ''     # el nombre, tal cual aparece en la tabla
respuesta_d = ''   # por qué

# (e) Con el modelo que elegiste ya en producción, ¿cómo te darías cuenta
#     de que empezó a andar peor?
respuesta_e = ''

# --- Validacion: ninguna respuesta puede quedar vacia ---
respuestas = {
    '(a) El acc_test mas alto': respuesta_a,
    '(b) El acc_train mas alto': respuesta_b,
    '(c) El acc_test mas bajo': respuesta_c,
    '(d) Cual pondrias en produccion (candidato)': candidato,
    '(d) Cual pondrias en produccion (por que)': respuesta_d,
    '(e) Como te darias cuenta de que empeora': respuesta_e,
}
vacias = [k for k, v in respuestas.items() if not v.strip()]
for etiqueta, texto in respuestas.items():
    print(f'  {etiqueta}: {"OK" if texto.strip() else "VACIA"}')
print()
if vacias:
    print('Completa las respuestas vacias y volve a correr esta celda.')
else:
    print('Las 5 respuestas estan completas.')
    print('Recorda: no se auto-corrigen, las lee el docente en tu PR.')

---
## ✅ Llegaste al final

Miraste siete candidatos, los consultaste desde el tracking y tomaste una decisión con la evidencia a la vista. Eso es, literalmente, el trabajo: el modelo lo entrena cualquiera, **promoverlo es una decisión que alguien firma**.

---

## 📦 Entrega

Generá tu archivo de entrega. **NO commitees el `.ipynb`** (es template compartido — generaría conflictos con el resto de los estudiantes). Reglas completas en [`README.md`](README.md).

> La entrega **relee los candidatos de MLflow** y escribe la tabla con lo que el tracking devuelve ahora: nada se autoreporta. Lo único que sale de vos son tus respuestas.
>
> **Las cinco respuestas son obligatorias**: si falta alguna, la celda no genera el archivo.

In [ ]:
# Compuestos PEGADOS, con la mayuscula adentro: "Maria Jose" -> MariaJose, "Garcia Lopez" -> GarciaLopez.
# De ahi salen tu archivo (garcialopez-mariajose.txt) y tu rama (estudiante/garcialopez-mariajose):
# UN solo guion, el que separa apellido de nombre. Si los dejas separados, el script pega los espacios igual.
nombre = ''    # <-- Completar con tu nombre
apellido = ''  # <-- Completar con tu apellido
# Los usa la Entrega: el nombre de tu .txt y el codigo de verificacion.

In [ ]:
# ============================================================
# Entrega: las metricas salen de MLflow, no de lo que escribas aca.
# ============================================================
import hashlib
import re
import subprocess
import unicodedata
from datetime import date
from pathlib import Path

# --- Prerrequisitos: esta celda depende de lo que dejaron las anteriores ---
# Sin esta guarda, el notebook pedia confirmacion, creaba la carpeta y recien
# despues fallaba con un NameError pelado.
_faltan = [_v for _v in ['CORTO', 'COLUMNAS', 'EXPERIMENTO', 'MODO', 'TRACKING', 'apellido',
                         'candidato', 'leer_candidatos', 'nombre', 'respuesta_a', 'respuesta_b',
                         'respuesta_c', 'respuesta_d', 'respuesta_e']
           if _v not in globals()]
if _faltan:
    raise RuntimeError(
        'Faltan datos de celdas anteriores: ' + ', '.join(_faltan) + '. '
        'Corre la Parte 1 (los candidatos) y la Parte 2 (las preguntas) antes de generar la entrega.'
    )

# Las cinco respuestas: sin ellas la entrega queda incompleta.
_sin_respuesta = [_v for _v in ['respuesta_a', 'respuesta_b', 'respuesta_c', 'candidato',
                                'respuesta_d', 'respuesta_e']
                  if not globals()[_v].strip()]
if _sin_respuesta:
    raise RuntimeError(
        'Faltan respuestas: ' + ', '.join(_sin_respuesta) + '. '
        'Completa las cinco preguntas de la Parte 2 y volve a correr esta celda.'
    )

if not nombre.strip() or not apellido.strip():
    raise ValueError('Completa tu nombre y apellido en la celda anterior antes de ejecutar.')

# --- En que rama estas: el aviso va ANTES de escribir, no despues ---
try:
    rama_actual = subprocess.check_output(
        ['git', 'branch', '--show-current'],
        stderr=subprocess.DEVNULL,
    ).decode().strip()
except Exception:
    rama_actual = '(no detectada)'

if rama_actual in ('main', 'master', 'dev', '', '(no detectada)'):
    print(f'ATENCION: estas en la rama {rama_actual!r}.')
    print('    Antes de subir tu entrega tenes que estar en TU rama personal')
    print('    estudiante/apellido-nombre (SIEMPRE la misma: tu PR se actualiza solo).')
    print('        git checkout estudiante/apellido-nombre   (reemplaza por tu rama)')
    print()
    print('    Igual generamos el archivo, pero acordate del checkout antes del push.')
    print()
else:
    print(f'OK -- rama actual: {rama_actual!r}.')
    print()

# Se relee el tracking: lo que va al .txt es lo que devuelve ahora.
_t = leer_candidatos()
_nombres = set(_t["candidato"])

estado_eleccion = 'OK' if candidato.strip() in _nombres else 'CANDIDATO_INEXISTENTE'

_filas = [
    f"  {r.candidato:<24} n_train={r.n_train:<4} n_test={r.n_test:<3} "
    f"acc_train={r.acc_train:.2f} acc_test={r.acc_test:.2f}"
    for r in _t.itertuples()
]
_huella = ';'.join(f"{r.candidato}:{r.acc_train:.2f}:{r.acc_test:.2f}" for r in _t.itertuples())
codigo = hashlib.sha256(
    f"{apellido.strip().lower()}-{nombre.strip().lower()}-veredicto-{MODO}-{_huella}"
    f"-{candidato.strip()}-{date.today().isoformat()}".encode()).hexdigest()[:12].upper()

print('=' * 56)
print('        ENTREGA - CLASE 06 (El veredicto)')
print('=' * 56)
print(f'Estudiante: {nombre.strip()} {apellido.strip()}')
print(f'  Tracking: {MODO} ({TRACKING})')
print(f'  Candidatos: {len(_t)}')
print(f'  Eleccion: {candidato.strip()}  [{estado_eleccion}]')
print(f'  Codigo: {codigo}')
print('=' * 56)


def slug(s):
    """Normaliza un nombre o apellido a una sola palabra.

    Los compuestos van PEGADOS: el guion se reserva como separador
    apellido-nombre, asi la rama nunca es ambigua.
        'Juan Pablo'   -> 'juanpablo'
        'Garcia Lopez' -> 'garcialopez'
    """
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    s = re.sub(r'[^a-zA-Z0-9]+', '', s).lower()   # sin espacios, guiones ni apostrofes
    return s


def _linea(s):
    """Una respuesta en una sola linea: un salto adentro cortaria el .txt."""
    return ' '.join(s.split())


filename = f'{slug(apellido)}-{slug(nombre)}.txt'
try:
    _raiz = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'],
                                         stderr=subprocess.DEVNULL).decode().strip())
    target = _raiz / 'clase06' / 'ejercicios' / 'estudiantes' / filename
except Exception:
    target = Path('estudiantes') / filename   # fallback: relativo a esta notebook

_verbo = 'SOBRESCRIBIR (ya existe)' if target.exists() else 'crear'
print(f'Voy a {_verbo}: clase06/ejercicios/estudiantes/{filename}')
try:
    confirm = input('Confirmas? (s/n): ').strip().lower()
except Exception:
    print('(entorno sin input interactivo -- se confirma automaticamente)')
    confirm = 's'

if confirm in ('s', 'si', 'y', 'yes'):
    target.parent.mkdir(parents=True, exist_ok=True)
    contenido = (
        f'Apellido: {apellido.strip()}\n'
        f'Nombre: {nombre.strip()}\n'
        f'Rama: {rama_actual}\n'
        f'Tracking: {MODO} ({TRACKING})\n'
        f'Experimento: {EXPERIMENTO}\n'
        f'Candidatos ({"leidos de MLflow" if mlflow is not None else "sin MLflow"}, '
        'no autoreportados):\n'
        + '\n'.join(_filas) + '\n'
        f'Elegido: {candidato.strip()} [{estado_eleccion}]\n'
        'Pregunta (a) - El acc_test mas alto:\n'
        f'  {_linea(respuesta_a)}\n'
        'Pregunta (b) - El acc_train mas alto:\n'
        f'  {_linea(respuesta_b)}\n'
        'Pregunta (c) - El acc_test mas bajo:\n'
        f'  {_linea(respuesta_c)}\n'
        'Pregunta (d) - Cual pondrias en produccion:\n'
        f'  {_linea(candidato)}\n'
        f'  {_linea(respuesta_d)}\n'
        'Pregunta (e) - Como te darias cuenta de que empeora:\n'
        f'  {_linea(respuesta_e)}\n'
        f'Codigo: {codigo}\n'
        f'Fecha: {date.today().isoformat()}\n'
    )
    target.write_text(contenido, encoding='utf-8')
    print()
    print(f'Archivo creado: clase06/ejercicios/estudiantes/{filename}')
    print()
    rama_ok = rama_actual.startswith('estudiante/')
    rama_push = rama_actual if rama_ok else 'estudiante/apellido-nombre'
    nota_rama = '' if rama_ok else '   <-- reemplaza por TU rama'
    print('Ahora subi SOLO ese archivo. Copia y pega estos tres comandos:')
    print()
    print(f'  git add clase06/ejercicios/estudiantes/{filename}')
    print('  git commit -m "clase06 (MachineLearning)"')
    print(f'  git push origin {rama_push}{nota_rama}')
else:
    print('No se escribio nada. Volve a correr esta celda cuando quieras confirmar.')

### 📦 Subí tu entrega

Subí **solo el `.txt`** (NO el `.ipynb`, que es template compartido).

**La celda de arriba te imprimió los tres comandos con tu archivo y tu rama ya puestos.** Copiálos de ahí y pegálos en la terminal, parado en la raíz del repo.

> **No uses `git add .`**: subirías también el `ejercicio.ipynb` modificado.

El `git push` actualiza tu PR abierto desde la Clase 01 — no abrís uno nuevo.

> Esta entrega es de **lectura y decisión**: no reemplaza al **TP Final**, que es el trabajo grande del cierre y se entrega la semana siguiente. Está en [`TpFinal/`](../../TpFinal/).